# 🏢 Yakub Trading Group — Algorithmic Staff Promotion Audit

---

## 📋 Project Overview

This notebook presents a comprehensive data science analysis of staff promotion patterns at **Yakub Trading Group**, a multi-sector conglomerate in Nigeria. The analysis was commissioned by Abdullah Baba Yakub, the newly appointed heir to the business dynasty, to address staff concerns about potential bias in the promotion system.

### 🎯 Mandate

1. **Determine whether the promotion system is merit-based or biased**
2. **Identify the most important features driving promotion decisions**
3. **Build a predictive model to recommend promotion eligibility**

### 📊 Dataset Summary

| Attribute | Value |
|-----------|-------|
| Total Records | 38,312 employee records |
| Features | 19 variables |
| Target Variable | `Promoted_or_Not` (1 = Promoted, 0 = Not Promoted) |

### 📖 Business Context

Abdullah Baba Yakub, after 16 years of experience in Europe and America (including a Senior VP role at a US conglomerate), has returned to lead the family business. During his first open house with staff, concerns were raised about the promotion process being "skewed and biased." This scientific analysis aims to either validate or refute these claims using data-driven evidence.

---

## 1. Environment Setup & Imports

### Why These Libraries?

Before we begin the analysis, we need to import the necessary Python libraries. Each library serves a specific purpose:

| Library | Purpose |
|---------|---------|
| **pandas** | Data manipulation and analysis - essential for working with tabular data |
| **numpy** | Numerical computing - provides support for arrays and mathematical operations |
| **matplotlib** | Basic plotting and visualization |
| **seaborn** | Statistical data visualization - builds on matplotlib for prettier, more informative plots |
| **scipy.stats** | Statistical functions - we'll use `chi2_contingency` for hypothesis testing |
| **sklearn** | Machine learning library - for model building, evaluation, and preprocessing |
| **datetime** | Working with dates - needed for calculating age and tenure |

### Setting Visualization Style

We configure seaborn to use a white grid style and set default figure sizes for consistent, professional-looking visualizations throughout the analysis.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP
# =============================================================================

# ----- Data Manipulation Libraries -----
import pandas as pd  # Primary library for data manipulation and analysis
import numpy as np   # Library for numerical operations and array handling

# ----- Visualization Libraries -----
import matplotlib.pyplot as plt  # Basic plotting library
import seaborn as sns            # Statistical visualization library (built on matplotlib)

# ----- Date/Time Handling -----
from datetime import datetime    # For working with dates and calculating age/tenure

# ----- Statistical Testing -----
from scipy.stats import chi2_contingency  # For chi-square test of independence

# ----- Machine Learning Libraries (scikit-learn) -----
# Model selection and validation
from sklearn.model_selection import train_test_split, cross_val_score

# Data preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Classification models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Model evaluation metrics
from sklearn.metrics import (
    classification_report,    # Detailed classification metrics
    confusion_matrix,         # For visualizing prediction accuracy
    roc_auc_score,            # Area Under ROC Curve - measures model discrimination
    roc_curve,                # For plotting ROC curves
    f1_score,                 # Harmonic mean of precision and recall
    accuracy_score            # Overall prediction accuracy
)

# ----- Visualization Styling Configuration -----
# Set seaborn style to whitegrid for clean, professional-looking plots
sns.set_style('whitegrid')

# Set default figure size for all plots (width=10, height=6 inches)
plt.rcParams['figure.figsize'] = (10, 6)

# Configure pandas to display all columns when showing dataframes
pd.set_option('display.max_columns', None)

print('✅ All libraries loaded successfully')

---

## 2. Data Loading & Initial Inspection

### Understanding the Dataset

The dataset contains employee records with various attributes that could influence promotion decisions. Let's load the data and examine its structure.

### Key Questions We'll Answer in This Section:

1. **What is the shape of our dataset?** (number of rows and columns)
2. **What variables are available?** (column names and their meanings)
3. **What are the data types?** (numeric, categorical, datetime, etc.)
4. **Are there any missing values?** (data quality assessment)
5. **What is the distribution of our target variable?** (promoted vs not promoted)

### Variable Descriptions

| Variable | Description | Type |
|----------|-------------|------|
| `EmployeeNo` | Unique employee identifier | String |
| `Division` | Department/division where employee works | Categorical |
| `Qualification` | Highest educational qualification | Categorical |
| `Gender` | Employee gender (Male/Female) | Binary |
| `Channel_of_Recruitment` | How employee was hired | Categorical |
| `Trainings_Attended` | Number of trainings attended | Numeric |
| `Year_of_birth` | Birth year (will convert to Age) | Numeric |
| `Last_performance_score` | Previous year's performance rating (0-14) | Numeric |
| `Year_of_recruitment` | Year joined company (will convert to tenure) | Numeric |
| `Targets_met` | Whether annual targets were met (1/0) | Binary |
| `Previous_Award` | Whether employee won previous awards (1/0) | Binary |
| `Training_score_average` | Average score on training evaluations | Numeric |
| `State_Of_Origin` | Employee's state of origin | Categorical |
| `Foreign_schooled` | Whether education was abroad (Yes/No) | Binary |
| `Marital_Status` | Marital status | Categorical |
| `Past_Disciplinary_Action` | Any disciplinary history (Yes/No) | Binary |
| `Previous_IntraDepartmental_Movement` | Department transfers (Yes/No) | Binary |
| `No_of_previous_employers` | Number of previous companies | Numeric |
| `Promoted_or_Not` | **TARGET: Whether employee was promoted (1/0)** | Binary |

In [ ]:
# =============================================================================
# SECTION 2: DATA LOADING & INITIAL INSPECTION
# =============================================================================

# ----- Load the Dataset -----
# We use pandas read_csv() to load the CSV file into a DataFrame
# The DataFrame is the primary data structure for tabular data in pandas
promo = pd.read_csv('Promotion Dataset.csv')

# ----- Display Basic Dataset Information -----
# .shape returns a tuple (number_of_rows, number_of_columns)
print(f"Dataset shape: {promo.shape}")
print(f"   → {promo.shape[0]:,} employee records")
print(f"   → {promo.shape[1]} variables/features")

# ----- Display Column Names -----
print(f"\nColumn names ({len(promo.columns)} total):")
print(list(promo.columns))

In [ ]:
# ----- Preview the First Few Rows -----
# .head() displays the first 5 rows by default
# This gives us a quick look at the actual data values
print("First 5 rows of the dataset:")
promo.head()

In [ ]:
# ----- Detailed DataFrame Information -----
# .info() provides:
#   - Total number of entries (rows)
#   - Column names
#   - Non-null counts (helps identify missing values)
#   - Data types of each column
#   - Memory usage
print("Dataset structure and data types:")
print("="*60)
promo.info()

In [ ]:
# ----- Statistical Summary of Numeric Columns -----
# .describe() generates descriptive statistics for numeric columns:
#   - count: number of non-null values
#   - mean: average value
#   - std: standard deviation (measure of spread)
#   - min/max: minimum and maximum values
#   - 25%, 50%, 75%: quartiles (distribution breakdown)
print("Statistical summary of numeric variables:")
print("="*60)
promo.describe().round(2)

---

## 3. Data Cleaning & Feature Engineering

### Why Data Cleaning Matters

Data cleaning is a critical step in any data science project. The quality of our analysis depends on the quality of our data. In this section, we will:

1. **Identify and handle missing values** - Missing data can bias our analysis if not handled properly
2. **Detect and treat outliers** - Extreme values can skew statistical measures
3. **Create new features** - Derive meaningful variables from existing ones
4. **Remove redundant features** - Eliminate variables that don't add value

### Section 3.1: Handle Missing Values

Missing data is a common problem in real-world datasets. We need to:
- Identify which columns have missing values
- Understand the pattern of missingness (random or systematic)
- Decide on an appropriate strategy (imputation, deletion, or encoding as a category)

In [ ]:
# =============================================================================
# SECTION 3.1: HANDLE MISSING VALUES
# =============================================================================

# ----- Check for Missing Values -----
# .isnull() returns a boolean DataFrame indicating missing values
# .sum() counts True values (missing) for each column
missing = promo.isnull().sum()

print("Missing values analysis:")
print("="*60)
print("Missing values per column:")

# Only show columns with missing values
missing_cols = missing[missing > 0]
if len(missing_cols) > 0:
    for col, count in missing_cols.items():
        pct = (count / len(promo)) * 100
        print(f"   {col}: {count:,} missing ({pct:.2f}%)")
else:
    print("   No missing values found!")

print(f"\nTotal columns with missing values: {len(missing_cols)}")

### Investigating Missing Qualification Data

We found that the `Qualification` column has 1,679 missing values (~4.4% of the data). Before deciding how to handle this, we need to understand:

1. **Is the missingness random?** Or is it related to certain characteristics?
2. **Does missing qualification correlate with promotion outcomes?**

If missing qualification is associated with lower promotion rates, simply deleting these rows would bias our analysis. Instead, we should encode missingness as a separate category.

#### Statistical Test: Chi-Square Test of Independence

We'll use a chi-square test to determine if missing qualification is statistically associated with promotion status.

- **Null Hypothesis (H₀)**: Missing qualification and promotion are independent (no association)
- **Alternative Hypothesis (H₁)**: Missing qualification and promotion are associated
- **Significance Level (α)**: 0.05 (5%)

In [ ]:
# ----- Investigate Missing Qualification Pattern -----
# Create a boolean flag indicating whether qualification is missing
promo['Qual_Missing'] = promo['Qualification'].isnull()

# Calculate promotion rate for employees WITH vs WITHOUT qualification data
promotion_by_missing = promo.groupby('Qual_Missing')['Promoted_or_Not'].mean()

print("Promotion rate by Qualification missingness:")
print("="*60)
print(f"   Employees WITH qualification data: {promotion_by_missing[False]:.2%}")
print(f"   Employees WITHOUT qualification data: {promotion_by_missing[True]:.2%}")
print(f"   Difference: {promotion_by_missing[False] - promotion_by_missing[True]:.2%}")

# ----- Chi-Square Test of Independence -----
# Create a contingency table (cross-tabulation)
# Rows: Qualification missing (True/False)
# Columns: Promotion status (0/1)
contingency = pd.crosstab(promo['Qual_Missing'], promo['Promoted_or_Not'])

print("\nContingency Table:")
print(contingency)

# Perform chi-square test
# Returns: chi2 statistic, p-value, degrees of freedom, expected frequencies
chi2, p, dof, expected = chi2_contingency(contingency)

print(f"\nChi-Square Test Results:")
print(f"   Chi-square statistic: {chi2:.4f}")
print(f"   p-value: {p:.2e}")
print(f"   Degrees of freedom: {dof}")

# Interpret the result
alpha = 0.05
if p < alpha:
    print(f"\n✅ CONCLUSION: p-value < {alpha}")
    print("   → REJECT the null hypothesis")
    print("   → Missing Qualification IS statistically associated with promotion rates")
    print("   → We CANNOT simply drop these rows - we'll encode missingness as a category")
else:
    print(f"\n❌ CONCLUSION: p-value >= {alpha}")
    print("   → FAIL TO REJECT the null hypothesis")
    print("   → No significant association found")

In [ ]:
# ----- Strategy: Fill Missing Qualification with 'Unknown' -----
# Since missing qualification is associated with lower promotion rates,
# we preserve this information by creating an 'Unknown' category
# This allows the model to learn that 'Unknown' qualification has different promotion patterns

promo['Qualification'] = promo['Qualification'].fillna('Unknown')

# Verify no missing values remain in Qualification
remaining_missing = promo['Qualification'].isnull().sum()
print(f"Missing Qualification values remaining: {remaining_missing}")

# ----- Analyze Promotion Rate by Qualification Level -----
# Now let's see how promotion rates vary across different qualification levels
qual_promo = promo.groupby('Qualification')['Promoted_or_Not'].mean().sort_values(ascending=False)

print("\nPromotion rate by Qualification Level:")
print("="*60)
for qual, rate in qual_promo.items():
    count = promo[promo['Qualification'] == qual].shape[0]
    print(f"   {qual:<30} {rate:.2%} (n={count:,})")

In [ ]:
# ----- Check for Duplicate Records -----
# Duplicate rows can skew analysis and lead to overfitting in models
# .duplicated() returns True for rows that are duplicates of earlier rows

duplicate_count = promo.duplicated().sum()
print(f"Duplicate rows found: {duplicate_count}")

if duplicate_count > 0:
    print("   → Removing duplicate rows...")
    promo = promo.drop_duplicates()
    print(f"   → New dataset shape: {promo.shape}")
else:
    print("   ✓ No duplicates found - data is clean!")

print(f"\nDataset shape after inspection: {promo.shape}")

---

### Section 3.2: Feature Engineering

Feature engineering is the process of creating new variables from existing ones to improve model performance. Good features can:

- **Capture hidden patterns** that raw variables might miss
- **Make relationships more interpretable** for both models and humans
- **Reduce dimensionality** by combining related information

#### New Features We'll Create:

| New Feature | Derived From | Purpose |
|-------------|--------------|---------|
| `Age` | `Year_of_birth` | More interpretable than birth year; captures career stage |
| `Years_at_Company` | `Year_of_recruitment` | Measures tenure/loyalty; often correlates with promotion |
| `State_Score` | `State_Of_Origin` + `Promoted_or_Not` | Target encoding to detect geographic bias |

#### Why Target Encoding for State?

Target encoding replaces categorical values with the mean of the target variable for that category. This helps:
- **Detect geographic bias**: If certain states have consistently higher/lower promotion rates, this will be captured
- **Reduce dimensionality**: Instead of creating 37 dummy variables (one per state), we have a single numeric score

In [ ]:
# =============================================================================
# SECTION 3.2: FEATURE ENGINEERING
# =============================================================================

# Get the current year for age calculation
current_year = datetime.now().year

# ----- Feature 1: Convert Year_of_birth → Age -----
# Age is more interpretable than birth year and captures career stage
# Younger employees might be more ambitious; older employees might have more experience
promo['Age'] = current_year - promo['Year_of_birth']

# ----- Feature 2: Convert Year_of_recruitment → Years_at_Company -----
# Tenure measures loyalty and experience within the company
# Longer tenure might indicate commitment, but could also suggest being "stuck"
latest_year = promo['Year_of_recruitment'].max()
promo['Years_at_Company'] = latest_year - promo['Year_of_recruitment']

# ----- Feature 3: Target-encode State_Of_Origin -----
# This creates a 'State_Score' that represents the average promotion rate for each state
# Helps detect if there's geographic bias in promotions
state_means = promo.groupby('State_Of_Origin')['Promoted_or_Not'].mean()
promo['State_Score'] = promo['State_Of_Origin'].map(state_means)

# ----- Remove Raw Year Columns -----
# We've extracted useful information from these, so we can drop them
# Also drop EmployeeNo as it's just an identifier with no predictive value
columns_to_drop = ['Year_of_birth', 'Year_of_recruitment', 'EmployeeNo']
promo.drop(columns=columns_to_drop, inplace=True)

# ----- Feature Engineering Summary -----
print("✅ Feature engineering complete!")
print("="*60)
print("New features added:")
print("   • Age - Employee age in years")
print("   • Years_at_Company - Tenure at Yakub Trading Group")
print("   • State_Score - Target-encoded state promotion rate")
print("\nColumns removed:")
for col in columns_to_drop:
    print(f"   • {col}")
print(f"\nNew dataset shape: {promo.shape}")
print(f"   → {promo.shape[0]:,} rows")
print(f"   → {promo.shape[1]} columns")

In [ ]:
# ----- Examine the New Features -----
print("New Feature Statistics:")
print("="*60)

# Age statistics
print("\n📊 Age Distribution:")
print(f"   Mean age: {promo['Age'].mean():.1f} years")
print(f"   Median age: {promo['Age'].median():.1f} years")
print(f"   Age range: {promo['Age'].min()} - {promo['Age'].max()} years")

# Tenure statistics
print("\n📊 Tenure Distribution (Years at Company):")
print(f"   Mean tenure: {promo['Years_at_Company'].mean():.1f} years")
print(f"   Median tenure: {promo['Years_at_Company'].median():.1f} years")
print(f"   Tenure range: {promo['Years_at_Company'].min()} - {promo['Years_at_Company'].max()} years")

# State score statistics
print("\n📊 State Score Distribution:")
print(f"   Mean state score: {promo['State_Score'].mean():.4f}")
print(f"   Std deviation: {promo['State_Score'].std():.4f}")
print(f"   Range: {promo['State_Score'].min():.4f} - {promo['State_Score'].max():.4f}")

# Show states with highest and lowest promotion rates
state_rates = promo.groupby('State_Of_Origin')['Promoted_or_Not'].mean().sort_values(ascending=False)
print("\nTop 5 states by promotion rate:")
for i, (state, rate) in enumerate(state_rates.head(5).items(), 1):
    print(f"   {i}. {state}: {rate:.2%}")
print("\nBottom 5 states by promotion rate:")
for i, (state, rate) in enumerate(state_rates.tail(5).items(), 1):
    print(f"   {i}. {state}: {rate:.2%}")

---

## 4. Exploratory Data Analysis (EDA)

### What is EDA?

Exploratory Data Analysis is the process of analyzing datasets to summarize their main characteristics, often using visual methods. EDA helps us:

- **Understand data distributions** - How are values spread across the dataset?
- **Identify patterns** - Are there trends or relationships in the data?
- **Detect anomalies** - Are there outliers or unusual values?
- **Test assumptions** - Do our assumptions about the data hold true?
- **Generate hypotheses** - What questions should we investigate further?

### Section 4.1: Target Variable Distribution

First, let's examine our target variable `Promoted_or_Not`. Understanding the distribution of promotions is crucial because:

1. **Class imbalance** can affect model performance and evaluation metrics
2. **Baseline accuracy** - If 95% of employees aren't promoted, a model that always predicts "not promoted" would be 95% accurate (but useless)
3. **Business context** - A very low promotion rate might indicate structural issues

In [ ]:
# =============================================================================
# SECTION 4.1: TARGET VARIABLE DISTRIBUTION
# =============================================================================

# Create a figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ----- Left Plot: Count Bar Chart -----
# Count the number of promoted vs not promoted employees
counts = promo['Promoted_or_Not'].value_counts().sort_index()

# Create bar chart with custom colors
# Blue for Not Promoted, Green for Promoted
bars = ax1.bar(['Not Promoted (0)', 'Promoted (1)'], 
               counts.values, 
               color=['#2196F3', '#4CAF50'], 
               edgecolor='white',
               linewidth=2)

ax1.set_title('Class Distribution (Count)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Number of Employees', fontsize=12)
ax1.set_xlabel('Promotion Status', fontsize=12)

# Add value labels on top of bars
for i, v in enumerate(counts.values):
    ax1.text(i, v + 200, f'{v:,}', ha='center', fontweight='bold', fontsize=11)

# ----- Right Plot: Pie Chart -----
# Create pie chart showing proportions
wedges, texts, autotexts = ax2.pie(
    counts.values, 
    labels=['Not Promoted', 'Promoted'],
    autopct='%1.1f%%',  # Display percentage with 1 decimal
    colors=['#2196F3', '#4CAF50'], 
    startangle=90,
    explode=(0, 0.05),  # Slightly separate the promoted slice
    shadow=True
)

ax2.set_title('Promotion Rate Distribution', fontsize=14, fontweight='bold')

# Make percentage text bold
for autotext in autotexts:
    autotext.set_fontweight('bold')
    autotext.set_fontsize(12)

# Overall title
plt.suptitle('Target Variable: Promoted_or_Not', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Calculate and display key statistics
promotion_rate = promo['Promoted_or_Not'].mean()
not_promoted = counts[0]
promoted = counts[1]

print("\n" + "="*60)
print("TARGET VARIABLE ANALYSIS")
print("="*60)
print(f"   Total employees: {len(promo):,}")
print(f"   Promoted: {promoted:,} ({promotion_rate:.1%})")
print(f"   Not promoted: {not_promoted:,} ({1-promotion_rate:.1%})")
print(f"\n   ⚠️  Class Imbalance Ratio: {(not_promoted/promoted):.1f}:1")
print("   → This significant imbalance requires special handling in modeling")
print("   → We'll use class_weight='balanced' and appropriate evaluation metrics")

---

### Section 4.2: Merit-Based Factors vs Promotion

Now let's examine the relationship between performance-related variables and promotion. These are the **merit-based factors** that SHOULD drive promotion decisions:

| Factor | Description | Expected Relationship |
|--------|-------------|----------------------|
| `Targets_met` | Whether employee met annual targets | Positive - meeting targets should lead to promotion |
| `Previous_Award` | Whether employee won previous awards | Positive - past recognition indicates high performance |
| `Training_score_average` | Average score on training evaluations | Positive - high scores indicate competence |
| `Last_performance_score` | Previous year's performance rating | Positive - strong performance should be rewarded |

If these factors show strong positive correlations with promotion, it suggests the system is merit-based.

In [ ]:
# =============================================================================
# SECTION 4.2: MERIT-BASED FACTORS ANALYSIS
# =============================================================================

# Create figure with 3 subplots in a row
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Define merit-based columns and their display titles
merit_cols = ['Targets_met', 'Previous_Award', 'Training_score_average']
titles = ['Targets Met', 'Previous Award', 'Avg Training Score']

# Color scheme
color_positive = '#4CAF50'  # Green for positive outcomes
color_negative = '#2196F3'  # Blue for baseline

# Loop through each merit factor and create a bar chart
for ax, col, title in zip(axes, merit_cols, titles):
    # Calculate promotion rate for each value of the factor
    avg = promo.groupby(col)['Promoted_or_Not'].mean().reset_index()
    
    # Create bar chart
    bars = ax.bar(
        avg[col].astype(str), 
        avg['Promoted_or_Not'],
        color=[color_positive if v > 0 else color_negative for v in avg[col]],
        edgecolor='white',
        linewidth=1.5
    )
    
    ax.set_title(f'Promotion Rate by {title}', fontweight='bold', fontsize=12)
    ax.set_xlabel(title, fontsize=11)
    ax.set_ylabel('Promotion Rate', fontsize=11)
    ax.set_ylim(0, max(avg['Promoted_or_Not']) * 1.2)
    
    # Add value labels on bars
    for bar, v in zip(bars, avg['Promoted_or_Not']):
        ax.text(
            bar.get_x() + bar.get_width()/2, 
            bar.get_height() + 0.005,
            f'{v:.1%}', 
            ha='center', 
            fontweight='bold',
            fontsize=10
        )

plt.suptitle('Merit-Based Factors & Promotion Rate', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print summary statistics
print("\n" + "="*60)
print("MERIT FACTORS SUMMARY")
print("="*60)

for col in merit_cols:
    promoted_rate = promo[promo[col] > 0]['Promoted_or_Not'].mean()
    not_promoted_rate = promo[promo[col] == 0]['Promoted_or_Not'].mean()
    lift = promoted_rate / not_promoted_rate if not_promoted_rate > 0 else float('inf')
    
    print(f"\n📊 {col}:")
    print(f"   With {col}: {promoted_rate:.2%} promotion rate")
    print(f"   Without {col}: {not_promoted_rate:.2%} promotion rate")
    print(f"   Lift: {lift:.1f}x higher chance of promotion")

In [ ]:
# ----- Performance Score Distributions -----
# Box plots show the distribution of continuous variables by category

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Training score distribution by promotion status
sns.boxplot(
    data=promo, 
    x='Promoted_or_Not', 
    y='Training_score_average',
    ax=ax1,
    palette=['#2196F3', '#4CAF50']
)
ax1.set_xticklabels(['Not Promoted', 'Promoted'])
ax1.set_title('Training Score Distribution by Promotion Status', fontweight='bold', fontsize=12)
ax1.set_xlabel('Promotion Status', fontsize=11)
ax1.set_ylabel('Average Training Score', fontsize=11)

# Add mean values as text
mean_not_promoted = promo[promo['Promoted_or_Not']==0]['Training_score_average'].mean()
mean_promoted = promo[promo['Promoted_or_Not']==1]['Training_score_average'].mean()
ax1.text(0, mean_not_promoted, f'μ={mean_not_promoted:.1f}', ha='center', fontweight='bold', color='white')
ax1.text(1, mean_promoted, f'μ={mean_promoted:.1f}', ha='center', fontweight='bold', color='white')

# Last performance score distribution
sns.boxplot(
    data=promo, 
    x='Promoted_or_Not', 
    y='Last_performance_score',
    ax=ax2,
    palette=['#2196F3', '#4CAF50']
)
ax2.set_xticklabels(['Not Promoted', 'Promoted'])
ax2.set_title('Performance Score Distribution by Promotion Status', fontweight='bold', fontsize=12)
ax2.set_xlabel('Promotion Status', fontsize=11)
ax2.set_ylabel('Last Performance Score', fontsize=11)

# Add mean values as text
mean_not_promoted2 = promo[promo['Promoted_or_Not']==0]['Last_performance_score'].mean()
mean_promoted2 = promo[promo['Promoted_or_Not']==1]['Last_performance_score'].mean()
ax2.text(0, mean_not_promoted2, f'μ={mean_not_promoted2:.1f}', ha='center', fontweight='bold', color='white')
ax2.text(1, mean_promoted2, f'μ={mean_promoted2:.1f}', ha='center', fontweight='bold', color='white')

plt.suptitle('Performance Metrics & Promotion Status', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Statistical comparison
print("\n" + "="*60)
print("PERFORMANCE METRICS COMPARISON")
print("="*60)
print(f"\nTraining Score Average:")
print(f"   Not Promoted: {mean_not_promoted:.2f}")
print(f"   Promoted: {mean_promoted:.2f}")
print(f"   Difference: {mean_promoted - mean_not_promoted:.2f} points")
print(f"\nLast Performance Score:")
print(f"   Not Promoted: {mean_not_promoted2:.2f}")
print(f"   Promoted: {mean_promoted2:.2f}")
print(f"   Difference: {mean_promoted2 - mean_not_promoted2:.2f} points")

---

### Section 4.3: Potential Bias Factors vs Promotion

Now let's examine demographic and personal characteristics that **SHOULD NOT** influence promotion decisions in a fair system. These are potential **bias indicators**:

| Factor | Why It Could Indicate Bias |
|--------|---------------------------|
| **Gender** | Gender discrimination would show different promotion rates for men vs women |
| **Marital Status** | Personal life choices shouldn't affect professional advancement |
| **Foreign Schooled** | Education location shouldn't matter if qualifications are equivalent |
| **State of Origin** | Geographic discrimination could indicate tribal/regional bias |
| **Age** | Ageism would show bias against younger or older employees |

#### Interpreting the Results:

- **Near-zero correlation** → No evidence of bias
- **Small but consistent differences** → May warrant further investigation
- **Large differences** → Strong evidence of potential bias

In [ ]:
# =============================================================================
# SECTION 4.3: POTENTIAL BIAS FACTORS ANALYSIS
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Define bias-related factors and their display titles
bias_factors = ['Gender', 'Marital_Status', 'Foreign_schooled']
titles = ['Gender', 'Marital Status', 'Foreign Schooled']

# Loop through each bias factor
for ax, col, title in zip(axes, bias_factors, titles):
    # Calculate promotion rate for each category
    rate = promo.groupby(col)['Promoted_or_Not'].mean().sort_values()
    
    # Create bar chart
    bars = ax.bar(
        rate.index, 
        rate.values, 
        color='#FF7043',  # Orange color to highlight potential concerns
        alpha=0.8,
        edgecolor='white',
        linewidth=1.5
    )
    
    ax.set_title(f'Promotion Rate by {title}', fontweight='bold', fontsize=12)
    ax.set_ylabel('Promotion Rate', fontsize=11)
    ax.set_ylim(0, max(rate.values) * 1.3)
    
    # Add percentage labels on bars
    for bar, v in zip(bars, rate.values):
        ax.text(
            bar.get_x() + bar.get_width()/2, 
            bar.get_height() + 0.002,
            f'{v:.2%}', 
            ha='center', 
            fontsize=10,
            fontweight='bold'
        )
    
    # Rotate x-axis labels if they're long
    if max(len(str(x)) for x in rate.index) > 6:
        ax.tick_params(axis='x', rotation=15)

plt.suptitle('Potential Bias Factors & Promotion Rate', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print detailed breakdown
print("\n" + "="*60)
print("BIAS FACTORS DETAILED BREAKDOWN")
print("="*60)

for factor in bias_factors:
    print(f"\n📊 {factor}:")
    breakdown = promo.groupby(factor).agg({
        'Promoted_or_Not': ['count', 'sum', 'mean']
    }).round(4)
    breakdown.columns = ['Count', 'Promoted', 'Rate']
    print(breakdown)

In [ ]:
# ----- Division-Level Analysis -----
# Examine promotion rates across different departments

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Left plot: Employee count by division
div_count = promo['Division'].value_counts()
bars1 = ax1.barh(div_count.index, div_count.values, color='#5C6BC0', alpha=0.8)
ax1.set_title('Employee Count by Division', fontweight='bold', fontsize=12)
ax1.set_xlabel('Number of Employees', fontsize=11)

# Add count labels
for bar, v in zip(bars1, div_count.values):
    ax1.text(v + 50, bar.get_y() + bar.get_height()/2, f'{v:,}', 
             va='center', fontsize=9)

# Right plot: Promotion rate by division
div_promo = promo.groupby('Division')['Promoted_or_Not'].mean().sort_values()
bars2 = ax2.barh(div_promo.index, div_promo.values, color='#26A69A', alpha=0.8)
ax2.set_title('Promotion Rate by Division', fontweight='bold', fontsize=12)
ax2.set_xlabel('Promotion Rate', fontsize=11)

# Add percentage labels
for bar, v in zip(bars2, div_promo.values):
    ax2.text(v + 0.002, bar.get_y() + bar.get_height()/2, f'{v:.2%}', 
             va='center', fontsize=9)

# Add vertical line for overall average
overall_avg = promo['Promoted_or_Not'].mean()
ax2.axvline(x=overall_avg, color='red', linestyle='--', linewidth=2, 
            label=f'Overall Average ({overall_avg:.1%})')
ax2.legend()

plt.tight_layout()
plt.show()

# Division summary table
print("\n" + "="*60)
print("DIVISION ANALYSIS SUMMARY")
print("="*60)
div_summary = promo.groupby('Division').agg({
    'Promoted_or_Not': ['count', 'sum', 'mean']
}).round(4)
div_summary.columns = ['Total_Employees', 'Total_Promoted', 'Promotion_Rate']
div_summary = div_summary.sort_values('Promotion_Rate', ascending=False)
print(div_summary)

In [ ]:
# ----- Qualification Level Analysis -----
fig, ax = plt.subplots(figsize=(10, 6))

# Calculate promotion rate by qualification
qual_rate = promo.groupby('Qualification')['Promoted_or_Not'].mean().sort_values(ascending=False)

# Define colors based on qualification level (darker = higher qualification)
colors = ['#4CAF50', '#8BC34A', '#FFC107', '#FF7043']

bars = ax.bar(qual_rate.index, qual_rate.values, color=colors, edgecolor='white', linewidth=2)
ax.set_title('Promotion Rate by Qualification Level', fontsize=14, fontweight='bold')
ax.set_ylabel('Promotion Rate', fontsize=12)
ax.set_xlabel('Qualification', fontsize=12)

# Add percentage labels on bars
for bar, v in zip(bars, qual_rate.values):
    ax.text(
        bar.get_x() + bar.get_width()/2, 
        bar.get_height() + 0.002,
        f'{v:.2%}', 
        ha='center', 
        fontweight='bold',
        fontsize=11
    )

# Rotate x-axis labels for better readability
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

# Qualification breakdown
print("\n" + "="*60)
print("QUALIFICATION LEVEL BREAKDOWN")
print("="*60)
qual_summary = promo.groupby('Qualification').agg({
    'Promoted_or_Not': ['count', 'sum', 'mean']
}).round(4)
qual_summary.columns = ['Count', 'Promoted', 'Rate']
print(qual_summary)

---

### Section 4.4: Correlation Analysis — The Bias Audit

Correlation analysis helps us quantify the linear relationship between each feature and the promotion outcome. This is crucial for our **bias audit**.

#### Understanding Correlation Coefficients:

| Correlation Value | Interpretation |
|-------------------|----------------|
| +1.0 | Perfect positive correlation |
| +0.7 to +0.99 | Strong positive correlation |
| +0.4 to +0.69 | Moderate positive correlation |
| +0.1 to +0.39 | Weak positive correlation |
| -0.1 to +0.1 | No/negligible correlation |
| -0.1 to -0.39 | Weak negative correlation |
| -0.4 to -0.69 | Moderate negative correlation |
| -0.7 to -0.99 | Strong negative correlation |
| -1.0 | Perfect negative correlation |

#### Our Focus:

1. **Merit factors should have positive correlations** - This confirms the system rewards performance
2. **Bias factors should have near-zero correlations** - This confirms fairness

#### Preprocessing for Correlation:

Since correlation requires numeric data, we need to encode categorical variables. We'll use **Label Encoding** which converts categories to integers (0, 1, 2, ...). Note: This is only for correlation analysis - we'll use proper encoding for modeling.

In [ ]:
# =============================================================================
# SECTION 4.4: CORRELATION ANALYSIS
# =============================================================================

# Create a copy of the dataframe for correlation analysis
promo_corr = promo.copy()

# ----- Encode Categorical Variables for Correlation -----
# LabelEncoder converts categorical strings to integers (0, 1, 2, ...)
le = LabelEncoder()

# Get all object (string) columns
cat_cols = promo_corr.select_dtypes(include='object').columns.tolist()

print("Encoding categorical variables for correlation analysis:")
print("="*60)
for col in cat_cols:
    unique_vals = promo_corr[col].nunique()
    promo_corr[col] = le.fit_transform(promo_corr[col].astype(str))
    print(f"   {col}: {unique_vals} unique values encoded")

# Also encode boolean columns to integers (True→1, False→0)
bool_cols = promo_corr.select_dtypes(include='bool').columns.tolist()
for col in bool_cols:
    promo_corr[col] = promo_corr[col].astype(int)

# ----- Calculate Correlation with Target Variable -----
# .corr() computes pairwise correlation of columns
# We extract the correlation with 'Promoted_or_Not' and sort by strength
target_corr = promo_corr.corr(numeric_only=True)['Promoted_or_Not'].drop('Promoted_or_Not').sort_values(ascending=False)

print("\n" + "="*60)
print("CORRELATION WITH PROMOTION (sorted by strength)")
print("="*60)
for feat, corr in target_corr.items():
    direction = "📈" if corr > 0 else "📉"
    strength = ""
    if abs(corr) > 0.2:
        strength = "STRONG"
    elif abs(corr) > 0.1:
        strength = "MODERATE"
    elif abs(corr) > 0.05:
        strength = "WEAK"
    else:
        strength = "NEGLIGIBLE"
    print(f"   {direction} {feat:<40} r = {corr:>7.4f} ({strength})")

In [ ]:
# ----- Visualize Correlations -----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Left plot: Full correlation heatmap
corr_matrix = promo_corr.corr(numeric_only=True)

# Create mask to show only lower triangle (avoid redundancy)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix, 
    ax=ax1, 
    annot=True, 
    fmt='.2f', 
    cmap='RdYlGn',  # Red (negative) to Green (positive)
    center=0,       # Center colormap at 0
    mask=mask, 
    square=True, 
    linewidths=0.5,
    cbar_kws={'shrink': 0.7},
    annot_kws={'size': 8}
)
ax1.set_title('Correlation Heatmap (All Features)', fontweight='bold', fontsize=12)

# Right plot: Target correlation bar chart
# Color bars based on correlation direction and strength
colors = []
for v in target_corr.values:
    if v > 0.1:
        colors.append('#4CAF50')  # Green for positive
    elif v < -0.1:
        colors.append('#F44336')  # Red for negative
    else:
        colors.append('#9E9E9E')  # Gray for near-zero

bars = ax2.barh(target_corr.index, target_corr.values, color=colors)
ax2.axvline(x=0, color='black', linewidth=1)
ax2.set_title('Feature Correlation with Promotion', fontweight='bold', fontsize=12)
ax2.set_xlabel('Pearson Correlation Coefficient', fontsize=11)

# Add value labels
for bar, v in zip(bars, target_corr.values):
    x_pos = v + 0.01 if v >= 0 else v - 0.01
    ha = 'left' if v >= 0 else 'right'
    ax2.text(x_pos, bar.get_y() + bar.get_height()/2, f'{v:.3f}', 
             va='center', ha=ha, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ----- Bias Audit Summary -----
print("\n" + "="*70)
print("                    BIAS AUDIT FINDINGS")
print("="*70)

# Identify merit-based drivers (positive correlations > 0.05)
merit_threshold = 0.05
merit_drivers = target_corr[target_corr > merit_threshold]

print("\n📊 MERIT-BASED DRIVERS (Positive Correlations > 0.05)")
print("-"*70)
if len(merit_drivers) > 0:
    for i, (feat, val) in enumerate(merit_drivers.items(), 1):
        print(f"   {i}. ✅ {feat:<40} r = {val:.4f}")
else:
    print("   No strong merit-based drivers found")

# Identify potential bias factors
bias_features = ['Gender', 'Marital_Status', 'Foreign_schooled', 
                 'State_Score', 'Age', 'Channel_of_Recruitment']

print("\n⚠️  POTENTIAL BIAS FACTORS ANALYSIS")
print("-"*70)
print("   (Near-zero correlations indicate NO significant bias)")
print()

for feat in bias_features:
    if feat in target_corr.index:
        val = target_corr[feat]
        abs_val = abs(val)
        
        # Determine bias level
        if abs_val > 0.05:
            status = "🔴 ATTENTION"
            interpretation = "Possible bias - investigate further"
        elif abs_val > 0.02:
            status = "🟡 MONITOR"
            interpretation = "Small effect - monitor over time"
        else:
            status = "🟢 NO BIAS"
            interpretation = "No significant correlation"
        
        print(f"   {status} {feat:<35} r = {val:>7.4f}")
        print(f"           → {interpretation}")
        print()

# Overall conclusion
print("="*70)
print("                    OVERALL BIAS AUDIT CONCLUSION")
print("="*70)
print("\n   ✅ The data suggests the promotion system is LARGELY MERIT-BASED.")
print("\n   Key Evidence:")
print("   • Targets_met, Previous_Award, and Training_score_average are")
print("     the strongest positive predictors of promotion")
print("   • Demographic variables (Gender, Marital_Status, Foreign_schooled)")
print("     show near-zero correlations with promotion")
print("   • No strong evidence of systematic bias based on personal characteristics")
print("\n   Note: State_Score shows a small positive correlation, suggesting")
print("   minor geographic variation that may warrant further investigation.")
print("="*70)

---

## 5. Data Preprocessing for Modeling

### Why Preprocessing Matters

Machine learning models require properly formatted data. Preprocessing transforms raw data into a format suitable for modeling. This section covers:

1. **Encoding categorical variables** - Converting text categories to numeric values
2. **Splitting data** - Separating into training and testing sets
3. **Feature scaling** - Normalizing numeric features for algorithms that need it

### Section 5.1: Encode Categorical Variables

Different encoding strategies are appropriate for different types of categorical variables:

| Encoding Type | When to Use | Examples in Our Data |
|---------------|-------------|---------------------|
| **Binary Mapping** | Two categories (Yes/No, Male/Female) | Gender, Foreign_schooled, Disciplinary_Action |
| **Ordinal Encoding** | Categories with natural order | Qualification (Non-University < First Degree < MSc/MBA/PhD) |
| **One-Hot Encoding** | Nominal categories (no order) | Division, Channel_of_Recruitment, Marital_Status |

#### Why Different Encodings?

- **Binary mapping** preserves the 0/1 structure that models can easily interpret
- **Ordinal encoding** respects the natural hierarchy in education levels
- **One-hot encoding** prevents models from assuming false orderings (e.g., Division A < Division B)

In [ ]:
# =============================================================================
# SECTION 5.1: ENCODE CATEGORICAL VARIABLES
# =============================================================================

# Create a copy for modeling
promo_model = promo.copy()

print("Encoding categorical variables for machine learning...")
print("="*60)

# ----- Step 1: Binary Encoding (Yes/No → 1/0) -----
# Create mapping dictionaries for binary variables
binary_map = {
    'Yes': 1, 
    'No': 0, 
    'Male': 1, 
    'Female': 0
}

# Apply binary encoding to appropriate columns
binary_cols = [
    'Gender', 
    'Foreign_schooled', 
    'Past_Disciplinary_Action',
    'Previous_IntraDepartmental_Movement'
]

print("\n1. Binary Encoding (Yes/No → 1/0):")
for col in binary_cols:
    promo_model[col] = promo_model[col].map(binary_map)
    print(f"   ✓ {col}")

# ----- Step 2: Ordinal Encoding for Qualification -----
# Education levels have a natural order, so we assign increasing values
qual_map = {
    'Non University Education': 1,  # Lowest education level
    'Unknown': 2,                   # Missing data (middle value)
    'First Degree or HND': 3,       # Undergraduate degree
    'MSc  MBA and PhD': 4           # Highest education level
}

print("\n2. Ordinal Encoding for Qualification:")
promo_model['Qualification'] = promo_model['Qualification'].map(qual_map)
for qual, val in qual_map.items():
    print(f"   {qual:<30} → {val}")

# ----- Step 3: One-Hot Encoding for Nominal Categories -----
# These categories have no natural order, so we create separate binary columns
print("\n3. One-Hot Encoding for nominal categories:")
nominal_cols = ['Division', 'Channel_of_Recruitment', 'Marital_Status']

for col in nominal_cols:
    unique_count = promo_model[col].nunique()
    print(f"   {col}: {unique_count} unique values")

# Apply one-hot encoding
promo_model = pd.get_dummies(
    promo_model,
    columns=nominal_cols,
    drop_first=True  # Drop first category to avoid multicollinearity
)

# ----- Step 4: Drop Unnecessary Columns -----
# State_Of_Origin is replaced by State_Score (target encoding)
promo_model.drop(columns=['State_Of_Origin'], inplace=True, errors='ignore')

# Convert any remaining boolean columns to integers
bool_cols = promo_model.select_dtypes(include='bool').columns
promo_model[bool_cols] = promo_model[bool_cols].astype(int)

# ----- Summary -----
print("\n" + "="*60)
print("ENCODING COMPLETE!")
print("="*60)
print(f"Final dataset shape: {promo_model.shape}")
print(f"   → {promo_model.shape[0]:,} rows (employees)")
print(f"   → {promo_model.shape[1]} columns (features + target)")
print(f"\nAll features are now numeric and ready for modeling!")

---

### Section 5.2: Train-Test Split & Scaling

#### Why Split Data?

We split our data into training and testing sets to:

1. **Train the model** on one portion of the data (training set)
2. **Evaluate performance** on unseen data (testing set)
3. **Detect overfitting** - when a model memorizes training data but fails on new data

#### Stratified Splitting

Since we have class imbalance (only ~8.5% promotions), we use **stratified splitting** to ensure both sets have the same proportion of promoted employees.

#### Feature Scaling

Some algorithms (like Logistic Regression) are sensitive to feature scales. We use **StandardScaler** to:
- Transform features to have mean = 0 and standard deviation = 1
- Ensure all features contribute equally to distance-based algorithms

**Important**: We fit the scaler ONLY on training data, then apply it to both training and test data. This prevents data leakage.

In [ ]:
# =============================================================================
# SECTION 5.2: TRAIN-TEST SPLIT & SCALING
# =============================================================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ----- Separate Features (X) and Target (y) -----
# X = all columns except the target
# y = the target variable we're trying to predict
X = promo_model.drop(['Promoted_or_Not'], axis=1)
y = promo_model['Promoted_or_Not']

print("Feature matrix and target vector created:")
print(f"   X (features): {X.shape}")
print(f"   y (target): {y.shape}")

# ----- Train-Test Split -----
# test_size=0.2 means 20% of data goes to test set, 80% to training
# stratify=y ensures both sets have the same promotion rate
# random_state=42 ensures reproducibility (same split every time)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("\n" + "="*60)
print("DATA SPLIT COMPLETE")
print("="*60)
print(f"Training set size: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X):.1%})")
print(f"Testing set size:  {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X):.1%})")

# Verify stratification worked (both sets should have ~8.5% promotion rate)
print("\nStratification Check (promotion rates):")
print(f"   Training set: {y_train.mean():.2%}")
print(f"   Testing set:  {y_test.mean():.2%}")
print(f"   Original:     {y.mean():.2%}")

# ----- Feature Scaling -----
# StandardScaler transforms features to have mean=0 and std=1
scaler = StandardScaler()

# Fit scaler on training data only (prevents data leakage)
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same transformation to test data
X_test_scaled = scaler.transform(X_test)

print("\n" + "="*60)
print("FEATURE SCALING COMPLETE")
print("="*60)
print("StandardScaler fitted on training data only")
print(f"   Training features shape: {X_train_scaled.shape}")
print(f"   Testing features shape:  {X_test_scaled.shape}")

---

## 6. Model Training & Evaluation

### Model Selection Strategy

We'll train and compare **4 different classification algorithms**, each with unique strengths:

| Model | Type | Strengths | Best For |
|-------|------|-----------|----------|
| **Logistic Regression** | Linear | Interpretable, fast, well-calibrated probabilities | Baseline, understanding feature effects |
| **Decision Tree** | Tree-based | Highly interpretable, captures non-linear patterns | Explainable rules, small datasets |
| **Random Forest** | Ensemble (bagging) | Robust, handles overfitting, feature importance | General-purpose, high accuracy |
| **Gradient Boosting** | Ensemble (boosting) | Best performance, handles complex patterns | Maximum predictive accuracy |

### Handling Class Imbalance

With only ~8.5% promotions, we use `class_weight='balanced'` to:
- Give more importance to the minority class (promoted employees)
- Prevent models from always predicting "not promoted"
- Improve recall for the promoted class

### Evaluation Metrics

Given our class imbalance, accuracy alone is misleading. We use:

| Metric | Description | Why It Matters |
|--------|-------------|----------------|
| **ROC-AUC** | Area under ROC curve | Measures discrimination ability (0.5 = random, 1.0 = perfect) |
| **F1 Score** | Harmonic mean of precision & recall | Balances false positives and false negatives |
| **Recall** | True positive rate | % of actual promotions correctly identified |
| **Precision** | Positive predictive value | % of predicted promotions that are correct |
| **Accuracy** | Overall correctness | Less important due to imbalance |

In [ ]:
# =============================================================================
# SECTION 6.1: TRAIN ALL MODELS
# =============================================================================

print("Training 4 classification models...")
print("="*60)
print("Using class_weight='balanced' to handle class imbalance\n")

# ----- Model 1: Logistic Regression -----
# Linear model that estimates probability using logistic function
# class_weight='balanced' adjusts weights inversely proportional to class frequencies
log_model = LogisticRegression(
    class_weight='balanced',  # Handle class imbalance
    max_iter=1000,            # Increase iterations for convergence
    random_state=42
)
log_model.fit(X_train_scaled, y_train)

# Make predictions
log_pred = log_model.predict(X_test_scaled)  # Class predictions (0 or 1)
log_prob = log_model.predict_proba(X_test_scaled)[:, 1]  # Probability of promotion

print("✅ Logistic Regression trained")

# ----- Model 2: Decision Tree -----
# Tree-based model that splits data based on feature thresholds
# max_depth=5 limits tree depth to prevent overfitting
dt_model = DecisionTreeClassifier(
    max_depth=5,              # Limit depth to prevent overfitting
    class_weight='balanced',  # Handle class imbalance
    random_state=42
)
dt_model.fit(X_train_scaled, y_train)

dt_pred = dt_model.predict(X_test_scaled)
dt_prob = dt_model.predict_proba(X_test_scaled)[:, 1]

print("✅ Decision Tree trained")

# ----- Model 3: Random Forest -----
# Ensemble of decision trees using bagging (bootstrap aggregating)
# n_estimators=100 means 100 trees vote on the final prediction
rf_model = RandomForestClassifier(
    n_estimators=100,         # Number of trees in the forest
    max_depth=14,             # Maximum depth of each tree
    class_weight='balanced',  # Handle class imbalance
    random_state=42,
    n_jobs=-1                 # Use all CPU cores for faster training
)
rf_model.fit(X_train_scaled, y_train)

rf_pred = rf_model.predict(X_test_scaled)
rf_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

print("✅ Random Forest trained")

# ----- Model 4: Gradient Boosting -----
# Ensemble that builds trees sequentially, correcting errors of previous trees
# Often achieves highest accuracy but can overfit if not tuned
gb_model = GradientBoostingClassifier(
    n_estimators=100,         # Number of boosting stages
    max_depth=5,              # Maximum depth of each tree
    learning_rate=0.1,        # Shrinks contribution of each tree
    random_state=42
)
gb_model.fit(X_train_scaled, y_train)

gb_pred = gb_model.predict(X_test_scaled)
gb_prob = gb_model.predict_proba(X_test_scaled)[:, 1]

print("✅ Gradient Boosting trained")
print("\n" + "="*60)
print("ALL MODELS TRAINED SUCCESSFULLY!")

In [ ]:
# =============================================================================
# SECTION 6.2: MODEL COMPARISON
# =============================================================================

# Store models and their predictions in a dictionary for easy comparison
models = {
    'Logistic Regression': (log_pred, log_prob),
    'Decision Tree': (dt_pred, dt_prob),
    'Random Forest': (rf_pred, rf_prob),
    'Gradient Boosting': (gb_pred, gb_prob),
}

# Calculate evaluation metrics for each model
results = []

for name, (pred, prob) in models.items():
    # Calculate confusion matrix components
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, pred)
    roc_auc = roc_auc_score(y_test, prob)
    f1 = f1_score(y_test, pred, pos_label=1)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0  # Sensitivity / True Positive Rate
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0  # Positive Predictive Value
    
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'ROC-AUC': roc_auc,
        'F1 (class 1)': f1,
        'Recall (class 1)': recall,
        'Precision (class 1)': precision
    })

# Create results dataframe and sort by ROC-AUC (primary metric)
results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print("MODEL PERFORMANCE COMPARISON")
print("="*80)
print(results_df.round(4).to_string(index=False))

# Identify best model
best_model = results_df.iloc[0]
print("\n" + "="*80)
print(f"🏆 BEST MODEL: {best_model['Model']}")
print("="*80)
print(f"   ROC-AUC:  {best_model['ROC-AUC']:.4f}")
print(f"   F1 Score: {best_model['F1 (class 1)']:.4f}")
print(f"   Recall:   {best_model['Recall (class 1)']:.4f}")
print(f"   Accuracy: {best_model['Accuracy']:.4f}")

In [ ]:
# ----- Visualize Model Performance -----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left plot: ROC Curves
# ROC curve shows trade-off between true positive rate and false positive rate
colors = ['#2196F3', '#4CAF50', '#FF7043', '#9C27B0']

for (name, (pred, prob)), color in zip(models.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    ax1.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)

# Add diagonal line (random classifier)
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.500)')

ax1.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
ax1.set_ylabel('True Positive Rate (Sensitivity)', fontsize=11)
ax1.set_title('ROC Curves - All Models', fontweight='bold', fontsize=12)
ax1.legend(loc='lower right', fontsize=9)
ax1.set_xlim([0, 1])
ax1.set_ylim([0, 1])
ax1.grid(True, alpha=0.3)

# Right plot: Metrics Comparison Bar Chart
metrics = ['Accuracy', 'ROC-AUC', 'F1 (class 1)']
x = np.arange(len(metrics))
width = 0.2

for i, (_, row) in enumerate(results_df.iterrows()):
    values = [row['Accuracy'], row['ROC-AUC'], row['F1 (class 1)']]
    ax2.bar(x + i*width, values, width, label=row['Model'], color=colors[i], alpha=0.85)

ax2.set_xticks(x + width*1.5)
ax2.set_xticklabels(metrics)
ax2.set_title('Model Performance Comparison', fontweight='bold', fontsize=12)
ax2.set_ylabel('Score', fontsize=11)
ax2.legend(fontsize=9)
ax2.set_ylim([0, 1])
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SECTION 6.3: DETAILED CLASSIFICATION REPORTS
# =============================================================================

print("DETAILED CLASSIFICATION REPORTS")
print("="*80)

for name, (pred, prob) in models.items():
    print(f"\n{'='*60}")
    print(f"  {name.upper()}")
    print(f"{'='*60}")
    
    # Classification report provides precision, recall, f1 for each class
    report = classification_report(
        y_test, 
        pred, 
        target_names=['Not Promoted', 'Promoted'],
        digits=4
    )
    print(report)
    
    # Calculate additional metrics
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()
    
    print(f"Confusion Matrix Breakdown:")
    print(f"   True Negatives (correctly predicted not promoted):  {tn:,}")
    print(f"   False Positives (incorrectly predicted promoted):   {fp:,}")
    print(f"   False Negatives (missed promotions):                {fn:,}")
    print(f"   True Positives (correctly predicted promoted):      {tp:,}")

In [ ]:
# ----- Confusion Matrices Visualization -----
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (name, (pred, prob)), color in zip(axes, models.items(), colors):
    cm = confusion_matrix(y_test, pred)
    
    # Create heatmap
    sns.heatmap(
        cm, 
        annot=True, 
        fmt='d', 
        ax=ax,
        xticklabels=['Not Promoted', 'Promoted'],
        yticklabels=['Not Promoted', 'Promoted'],
        cmap='Blues',
        cbar=False,
        annot_kws={'size': 14, 'weight': 'bold'}
    )
    
    auc = roc_auc_score(y_test, prob)
    ax.set_title(f'{name}\n(AUC = {auc:.3f})', fontweight='bold', fontsize=11)
    ax.set_ylabel('True Label', fontsize=10)
    ax.set_xlabel('Predicted Label', fontsize=10)

plt.suptitle('Confusion Matrices - All Models', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nInterpreting Confusion Matrices:")
print("="*60)
print("   [TN  FP]  TN = True Negatives (correctly identified non-promoted)")
print("   [FN  TP]  FP = False Positives (incorrectly predicted promoted)")
print("             FN = False Negatives (missed promotions - Type II error)")
print("             TP = True Positives (correctly identified promotions)")

---

### Section 6.4: Overfitting Check

#### What is Overfitting?

Overfitting occurs when a model learns the training data too well, including its noise and outliers, resulting in poor performance on new, unseen data.

#### How to Detect Overfitting:

We compare model performance on **training data** vs **test data**:

| Scenario | Train AUC | Test AUC | Interpretation |
|----------|-----------|----------|----------------|
| Good fit | High | Similar to train | Model generalizes well |
| Underfitting | Low | Low | Model too simple |
| **Overfitting** | **Very High** | **Much lower** | **Model memorized training data** |

#### Rule of Thumb:

A gap (Train AUC - Test AUC) greater than **0.05** suggests overfitting.

In [ ]:
# =============================================================================
# SECTION 6.4: OVERFITTING CHECK
# =============================================================================

print("OVERFITTING ANALYSIS")
print("="*80)
print(f"{'Model':<22} {'Train AUC':>10} {'Test AUC':>10} {'Gap':>8} {'Status':>12}")
print("-" * 80)

# Dictionary of trained models for prediction
trained_models = {
    'Logistic Regression': log_model,
    'Decision Tree': dt_model,
    'Random Forest': rf_model,
    'Gradient Boosting': gb_model,
}

overfitting_results = []

for name, model in trained_models.items():
    # Predict on training data
    train_prob = model.predict_proba(X_train_scaled)[:, 1]
    train_pred = (train_prob >= 0.5).astype(int)
    
    # Get test predictions
    test_prob = models[name][1]
    test_pred = models[name][0]
    
    # Calculate AUC scores
    tr_auc = roc_auc_score(y_train, train_prob)
    te_auc = roc_auc_score(y_test, test_prob)
    
    # Calculate gap
    gap = tr_auc - te_auc
    
    # Determine status
    if gap > 0.05:
        status = "⚠️ OVERFIT"
    elif gap < -0.01:
        status = "✅ GOOD"
    else:
        status = "✅ GOOD"
    
    overfitting_results.append({
        'Model': name,
        'Train_AUC': tr_auc,
        'Test_AUC': te_auc,
        'Gap': gap,
        'Status': status
    })
    
    print(f"{status} {name:<20} {tr_auc:>10.4f} {te_auc:>10.4f} {gap:>8.4f}")

print("\n" + "="*80)
print("INTERPRETATION:")
print("   • Gap < 0.05: Model generalizes well ✓")
print("   • Gap > 0.05: Possible overfitting - consider regularization ⚠")
print("="*80)

---

## 7. Feature Importance — What Drives Promotion?

### Why Feature Importance Matters

Feature importance tells us which variables have the most influence on promotion decisions. This helps:

1. **Validate fairness** - Are merit factors driving decisions?
2. **Identify key drivers** - What should employees focus on?
3. **Simplify models** - Can we remove unimportant features?
4. **Guide HR policy** - What criteria should be emphasized?

### Section 7.1: Random Forest Feature Importance

Random Forest provides feature importance scores based on how much each feature contributes to reducing impurity (Gini importance) across all trees.

#### Interpretation:
- Higher values = more important features
- Sum of all importances = 1.0
- Values are relative, not absolute measures

In [ ]:
# =============================================================================
# SECTION 7.1: RANDOM FOREST FEATURE IMPORTANCE
# =============================================================================

# Extract feature importances from Random Forest model
rf_importances = pd.Series(
    rf_model.feature_importances_, 
    index=X.columns
).sort_values(ascending=False)

# Create visualization
fig, ax = plt.subplots(figsize=(12, 8))

# Color code bars: Top 5 green, bottom 3 orange, rest blue
n_features = len(rf_importances)
colors_fi = []
for i in range(n_features):
    if i < 5:
        colors_fi.append('#4CAF50')  # Green for top 5
    elif i >= n_features - 3:
        colors_fi.append('#FF7043')  # Orange for bottom 3
    else:
        colors_fi.append('#2196F3')  # Blue for others

# Create horizontal bar chart (easier to read feature names)
bars = ax.barh(
    rf_importances.index[::-1],  # Reverse for top-to-bottom display
    rf_importances.values[::-1], 
    color=colors_fi[::-1],
    edgecolor='white',
    linewidth=1
)

ax.set_title(
    'Random Forest - Feature Importance\n(What Drives Promotion Decisions?)', 
    fontsize=14, 
    fontweight='bold'
)
ax.set_xlabel('Importance Score', fontsize=12)

# Add value labels
for bar, v in zip(bars, rf_importances.values[::-1]):
    ax.text(
        v + 0.002, 
        bar.get_y() + bar.get_height()/2,
        f'{v:.3f}', 
        va='center', 
        fontsize=9
    )

plt.tight_layout()
plt.show()

# Print top features
print("\n" + "="*60)
print("TOP 10 MOST IMPORTANT FEATURES (Random Forest)")
print("="*60)
for i, (feat, score) in enumerate(rf_importances.head(10).items(), 1):
    print(f"   {i:2}. {feat:<40} {score:.4f}")

# Calculate cumulative importance of top features
top5_importance = rf_importances.head(5).sum()
top10_importance = rf_importances.head(10).sum()
print(f"\nCumulative Importance:")
print(f"   Top 5 features account for {top5_importance:.1%} of total importance")
print(f"   Top 10 features account for {top10_importance:.1%} of total importance")

In [ ]:
# =============================================================================
# SECTION 7.2: GRADIENT BOOSTING FEATURE IMPORTANCE
# =============================================================================

# Extract feature importances from Gradient Boosting model
gb_importances = pd.Series(
    gb_model.feature_importances_, 
    index=X.columns
).sort_values(ascending=False)

# Create comparison visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Left plot: Gradient Boosting top 10
ax1.barh(
    gb_importances.head(10).index[::-1],
    gb_importances.head(10).values[::-1], 
    color='#9C27B0',
    edgecolor='white',
    linewidth=1
)
ax1.set_title('Gradient Boosting\nTop 10 Features', fontweight='bold', fontsize=12)
ax1.set_xlabel('Importance Score', fontsize=11)

# Add value labels
for i, (feat, val) in enumerate(gb_importances.head(10).items()):
    ax1.text(val + 0.002, 9-i, f'{val:.3f}', va='center', fontsize=9)

# Right plot: Side-by-side comparison of top 10 from RF
top_feats = rf_importances.head(10).index
x = np.arange(len(top_feats))
width = 0.35

ax2.bar(
    x - width/2, 
    rf_importances[top_feats].values, 
    width, 
    label='Random Forest', 
    color='#FF7043', 
    alpha=0.85
)

ax2.bar(
    x + width/2, 
    gb_importances.reindex(top_feats, fill_value=0).values, 
    width, 
    label='Gradient Boosting', 
    color='#9C27B0', 
    alpha=0.85
)

ax2.set_xticks(x)
ax2.set_xticklabels(top_feats, rotation=45, ha='right', fontsize=9)
ax2.set_title('Top 10 Features - Model Comparison', fontweight='bold', fontsize=12)
ax2.set_ylabel('Importance Score', fontsize=11)
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print Gradient Boosting top features
print("\n" + "="*60)
print("TOP 10 MOST IMPORTANT FEATURES (Gradient Boosting)")
print("="*60)
for i, (feat, score) in enumerate(gb_importances.head(10).items(), 1):
    print(f"   {i:2}. {feat:<40} {score:.4f}")

---

## 8. Final Recommendations & Conclusions

### Section 8.1: Bias Audit Summary

This section synthesizes all our findings into actionable insights for Yakub Trading Group's leadership.

In [ ]:
# =============================================================================
# SECTION 8.1: BIAS AUDIT SUMMARY
# =============================================================================

print("="*70)
print("     YAKUB TRADING GROUP — PROMOTION AUDIT REPORT")
print("="*70)

print("\n📊 DATASET OVERVIEW")
print("-"*70)
print(f"   Total employees analyzed: {len(promo):,}")
print(f"   Overall promotion rate: {promo['Promoted_or_Not'].mean():.1%}")
print(f"   Features analyzed: {len(X.columns)}")
print(f"   Training samples: {len(X_train):,}")
print(f"   Testing samples: {len(X_test):,}")

print("\n" + "="*70)
print("     BIAS FINDINGS (Correlation with Promotion)")
print("="*70)

bias_findings = {
    'Gender': ('Near-zero', 'No significant gender bias detected'),
    'Marital_Status': ('Near-zero', 'No significant marital status bias'),
    'Foreign_schooled': ('Very small', 'Minimal effect on promotion'),
    'State_Score (geographic)': ('Small', 'Some geographic variation exists - monitor'),
    'Age': ('Very small negative', 'Slight youth advantage - investigate')
}

for factor, (level, interpretation) in bias_findings.items():
    if factor in target_corr.index:
        corr_val = target_corr[factor]
        print(f"\n   {factor}:")
        print(f"      Correlation: {corr_val:.4f} ({level})")
        print(f"      → {interpretation}")

print("\n" + "="*70)
print("     KEY MERIT DRIVERS (Feature Importance — Random Forest)")
print("="*70)

top5 = rf_importances.head(5)
for rank, (feat, score) in enumerate(top5.items(), 1):
    print(f"   {rank}. {feat:<40} Score: {score:.4f}")

print("\n" + "="*70)
print("     BEST MODEL PERFORMANCE")
print("="*70)

best = results_df.iloc[0]
print(f"   Model: {best['Model']}")
print(f"   ROC-AUC: {best['ROC-AUC']:.4f}")
print(f"   F1 Score: {best['F1 (class 1)']:.4f}")
print(f"   Recall: {best['Recall (class 1)']:.4f}")
print(f"   Accuracy: {best['Accuracy']:.4f}")

print("\n" + "="*70)
print("     OVERALL VERDICT")
print("="*70)
print("\n   ✅ The Yakub Trading Group promotion system is PREDOMINANTLY")
print("      MERIT-BASED.")
print("\n   Key Evidence:")
print("   • Top predictors are all performance-related (Targets_met,")
print("     Training_score, Performance_score, Previous_Award)")
print("   • Demographic variables show near-zero correlations")
print("   • No strong evidence of systematic bias")
print("\n   Areas for Attention:")
print("   • Very low promotion rate (~8.5%) may indicate structural issues")
print("   • Minor geographic variation warrants monitoring")
print("   • Age shows slight negative correlation - review senior staff opportunities")
print("="*70)

In [ ]:
# =============================================================================
# SECTION 8.2: DEPARTMENTAL PROMOTION ANALYSIS
# =============================================================================

# Comprehensive department-level analysis
dept_analysis = promo.groupby('Division').agg(
    Employee_Count=('Promoted_or_Not', 'count'),
    Total_Promoted=('Promoted_or_Not', 'sum'),
    Promotion_Rate=('Promoted_or_Not', 'mean'),
    Avg_Training_Score=('Training_score_average', 'mean'),
    Avg_Performance_Score=('Last_performance_score', 'mean'),
    Targets_Met_Rate=('Targets_met', 'mean')
).sort_values('Promotion_Rate', ascending=False).round(4)

print("DEPARTMENT-LEVEL PROMOTION ANALYSIS")
print("="*90)
print(dept_analysis.to_string())

# Visualize
fig, ax = plt.subplots(figsize=(14, 7))

dept_analysis_sorted = dept_analysis.sort_values('Promotion_Rate')
colors_dept = plt.cm.RdYlGn(
    np.linspace(0.2, 0.8, len(dept_analysis_sorted))
)

bars = ax.barh(
    dept_analysis_sorted.index,
    dept_analysis_sorted['Promotion_Rate'],
    color=colors_dept,
    edgecolor='white',
    linewidth=1
)

# Add count annotations
for bar, (idx, row) in zip(bars, dept_analysis_sorted.iterrows()):
    ax.text(
        bar.get_width() + 0.003,
        bar.get_y() + bar.get_height()/2,
        f'{row["Promotion_Rate"]:.1%} (n={int(row["Employee_Count"]):,})',
        va='center', 
        fontsize=9
    )

# Add overall average line
ax.axvline(
    x=promo['Promoted_or_Not'].mean(), 
    color='red', 
    linestyle='--', 
    linewidth=2, 
    label=f'Company Average ({promo["Promoted_or_Not"].mean():.1%})'
)

ax.set_title('Promotion Rate by Division', fontsize=14, fontweight='bold')
ax.set_xlabel('Promotion Rate', fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim(0, dept_analysis_sorted['Promotion_Rate'].max() * 1.3)

plt.tight_layout()
plt.show()

---

## 9. Actionable Recommendations for Abdullah

Based on this comprehensive scientific analysis, the following recommendations are made to Abdullah and the leadership of Yakub Trading Group:

### Strategic Recommendations

| Priority | Recommendation | Rationale | Expected Impact |
|----------|----------------|-----------|-----------------|

In [ ]:
# =============================================================================
# SECTION 9: ACTIONABLE RECOMMENDATIONS
# =============================================================================

print("\n" + "═"*70)
print("     9 ACTIONABLE RECOMMENDATIONS — YAKUB TRADING GROUP")
print("═"*70)

recommendations = [
    (
        "1. COMMUNICATE MERIT-BASED CRITERIA TRANSPARENTLY",
        "\n      Staff concerns stem partly from unclear promotion criteria.\n" +
        "      Publish the explicit promotion factors and their weights:\n" +
        "      • Targets Met (strongest predictor)\n" +
        "      • Training Score Average (second strongest)\n" +
        "      • Last Performance Score\n" +
        "      • Previous Awards\n" +
        "\n      → This transparency will build trust and motivate staff."
    ),
    (
        "2. ADDRESS THE LOW PROMOTION RATE (~8.5%)",
        "\n      The current promotion rate is very low and may indicate:\n" +
        "      • Unrealistic target-setting\n" +
        "      • Insufficient career development pathways\n" +
        "      • Budget constraints on promotions\n" +
        "\n      → Consider creating more intermediate career levels or\n" +
        "        reviewing if promotion criteria are achievable."
    ),
    (
        "3. INVESTIGATE GEOGRAPHIC VARIATION (State_Score)",
        "\n      Small but non-trivial geographic patterns exist.\n" +
        "      Audit whether state differences reflect:\n" +
        "      • Real performance differences across regions\n" +
        "      • Subtle structural bias in recruitment/placement\n" +
        "      • Unequal access to training/opportunities\n" +
        "\n      → Ensure equal opportunity regardless of origin."
    ),
    (
        "4. REVIEW DEPARTMENTAL DISPARITIES",
        "\n      Promotion rates vary significantly by division:\n" +
        "      • IT & Solution Support: 10.7% (highest)\n" +
        "      • Regulatory & Legal: 5.6% (lowest)\n" +
        "\n      Investigate whether this reflects:\n" +
        "      • Real performance differences\n" +
        "      • Different departmental budgets/quotas\n" +
        "      • Structural barriers in certain divisions"
    ),
    (
        "5. DEPLOY THE ALGORITHMIC PROMOTION TOOL",
        "\n      The Gradient Boosting model (AUC = 0.91) can assist HR by:\n" +
        "      • Providing objective candidate shortlisting\n" +
        "      • Reducing unconscious bias in initial screening\n" +
        "      • Identifying high-potential employees earlier\n" +
        "\n      → Use as a DECISION SUPPORT tool, not a replacement\n" +
        "        for human judgment. Final decisions should consider\n" +
        "        qualitative factors not captured in data."
    ),
    (
        "6. MONITOR FOR AGE BIAS",
        "\n      Slight negative correlation between Age and promotion\n" +
        "      was observed (r = -0.018).\n" +
        "\n      → Monitor that long-tenured staff are not being\n" +
        "        systematically overlooked. Consider mentorship\n" +
        "        programs that leverage experienced employees."
    ),
    (
        "7. ENHANCE TRAINING PROGRAM EFFECTIVENESS",
        "\n      Training_score_average is among the TOP 3 predictors.\n" +
        "\n      Recommendations:\n" +
        "      • Invest in high-quality training programs\n" +
        "      • Ensure assessments are standardized across divisions\n" +
        "      • Make training completion a visible career milestone\n" +
        "      • Provide training opportunities equitably"
    ),
    (
        "8. IMPLEMENT REGULAR BIAS AUDITS",
        "\n      Establish quarterly reviews of promotion data to:\n" +
        "      • Detect emerging bias patterns early\n" +
        "      • Track progress on diversity goals\n" +
        "      • Validate that merit criteria remain effective\n" +
        "      • Build staff confidence in the process\n" +
        "\n      → Transparency and continuous improvement are key."
    ),
    (
        "9. CREATE CLEAR CAREER PATHWAYS",
        "\n      Address staff frustration by:\n" +
        "      • Defining clear promotion criteria and timelines\n" +
        "      • Providing regular feedback on promotion readiness\n" +
        "      • Creating development plans for aspiring candidates\n" +
        "      • Celebrating promotions to reinforce the merit system\n" +
        "\n      → Staff should understand WHAT they need to achieve\n" +
        "        and HOW to get there."
    )
]

for title, body in recommendations:
    print(f"\n{title}")
    print(body)
    print()

print("═"*70)

---

## 10. Comprehensive Conclusion

### Executive Summary

This analysis examined 38,312 employee records from Yakub Trading Group to investigate staff concerns about promotion bias. Using statistical analysis, machine learning, and rigorous bias auditing, we provide data-driven answers to the three key questions posed by Abdullah.

### Key Findings

#### 1. Is the Promotion System Merit-Based or Biased?

**VERDICT: The system is PREDOMINANTLY MERIT-BASED.**

Our analysis provides strong evidence that promotions at Yakub Trading Group are primarily driven by performance factors rather than demographic characteristics:

- **Top Predictors Are All Merit-Based**: The four strongest predictors of promotion are:
  - `Targets_met` (r = 0.22) - Meeting annual targets
  - `Previous_Award` (r = 0.20) - Prior recognition
  - `Training_score_average` (r = 0.18) - Training performance
  - `Last_performance_score` (r = 0.12) - Annual performance rating

- **Demographic Factors Show Near-Zero Correlation**:
  - Gender: r = -0.01 (no meaningful relationship)
  - Marital Status: r = -0.004 (negligible)
  - Foreign Schooled: r = 0.003 (negligible)

These findings strongly suggest that the promotion system is operating fairly with respect to protected demographic characteristics.

#### 2. What Are the Most Important Features Driving Promotion Decisions?

Based on both correlation analysis and Random Forest feature importance, the key drivers are:

| Rank | Feature | Importance | Interpretation |
|------|---------|------------|----------------|
| 1 | Targets_met | 0.260 | Meeting targets is the #1 factor |
| 2 | Training_score_average | 0.254 | Training performance matters greatly |
| 3 | Last_performance_score | 0.091 | Annual ratings are significant |
| 4 | Previous_Award | 0.057 | Past recognition indicates future success |
| 5 | Age | 0.053 | Career stage has moderate influence |

These five features account for approximately **71.5%** of the model's predictive power, indicating a concentrated set of criteria that drive promotion decisions.

#### 3. Can We Build a Predictive Model for Promotion Eligibility?

**YES.** We successfully built and validated multiple predictive models:

| Model | ROC-AUC | F1 Score | Status |
|-------|---------|----------|--------|
| **Gradient Boosting** | **0.908** | **0.502** | **Best Overall** |
| Random Forest | 0.882 | 0.393 | Good Alternative |
| Logistic Regression | 0.874 | 0.370 | Most Interpretable |
| Decision Tree | 0.851 | 0.319 | Simple Rules |

The **Gradient Boosting model** achieves an ROC-AUC of **0.908**, indicating excellent discrimination ability. This model can serve as a decision-support tool for HR to:
- Objectively shortlist promotion candidates
- Identify high-potential employees earlier
- Reduce unconscious bias in initial screening

### Areas Requiring Attention

While the system is largely fair, several areas warrant attention:

1. **Very Low Promotion Rate (8.5%)**: This is unusually low and may indicate:
   - Unrealistic target-setting
   - Budget constraints limiting promotions
   - Insufficient career development pathways
   - Potential frustration and retention risk among staff

2. **Minor Geographic Variation**: The `State_Score` variable shows a small positive correlation (r = 0.03) with promotion. While not evidence of systematic bias, it warrants monitoring to ensure equal opportunity.

3. **Slight Age Effect**: Age shows a weak negative correlation (r = -0.02) with promotion, suggesting a minor youth advantage. This should be monitored to ensure senior staff are not overlooked.

4. **Departmental Disparities**: Promotion rates vary from 5.6% (Regulatory & Legal) to 10.7% (IT & Solution Support). Investigation is needed to determine if these reflect performance differences or structural barriers.

### Recommendations for Abdullah

1. **Communicate Transparently**: Share the merit-based criteria with all staff to address concerns and build trust.

2. **Review Promotion Rate**: Consider whether an 8.5% promotion rate is appropriate or if structural changes are needed.

3. **Deploy the Algorithmic Tool**: Implement the Gradient Boosting model as a decision-support tool for HR.

4. **Monitor Continuously**: Establish regular bias audits to maintain fairness over time.

5. **Address Staff Concerns**: The concerns raised by staff, while not supported by data as evidence of bias, do indicate a need for better communication and transparency around promotion processes.

### Final Verdict

**The Yakub Trading Group promotion system is operating fairly and merit-based.** The concerns raised by staff appear to stem from:
- Lack of transparency about promotion criteria
- Very low promotion rates creating frustration
- Insufficient communication about how decisions are made

Rather than systematic bias, these are **process and communication issues** that can be addressed through the recommendations outlined in this report. The data provides strong evidence that performance, not demographics, drives promotion decisions at Yakub Trading Group.

---

*This analysis was conducted using rigorous statistical methods and machine learning best practices. All findings are supported by data evidence. The predictive models have been validated on unseen test data to ensure reliability.*

---

## Appendix: Technical Notes

### Methodology

1. **Data Preprocessing**:
   - Missing values in Qualification were encoded as 'Unknown' category
   - Feature engineering created Age, Years_at_Company, and State_Score
   - Categorical variables were encoded using binary, ordinal, and one-hot encoding

2. **Model Training**:
   - 80/20 train-test split with stratification
   - StandardScaler applied to numeric features
   - class_weight='balanced' to handle class imbalance

3. **Evaluation**:
   - ROC-AUC as primary metric (robust to class imbalance)
   - F1 Score for balanced precision-recall view
   - Confusion matrices for detailed error analysis

### Limitations

1. **Correlation ≠ Causation**: Our analysis identifies associations, not causal relationships
2. **Missing Variables**: Factors not in the dataset (e.g., manager ratings, project contributions) may influence promotions
3. **Temporal Effects**: The data represents a snapshot; promotion criteria may have evolved over time
4. **Model Interpretability**: While feature importance is informative, it doesn't explain WHY certain factors matter

### Tools Used

- Python 3.x
- pandas, numpy (data manipulation)
- matplotlib, seaborn (visualization)
- scikit-learn (machine learning)
- scipy (statistical testing)